# Match metadata: interactive sandbox

Use this notebook for bounded exploratory runs only.
Do not treat its outputs as production sessions.


## Baseline similarity matching (RR monthly vs ensemble consensus, DuckDB/Parquet)

Every daily transcription (ensemble file) is matched to the Rainfall-Rescue
monthly records by comparing month-by-month values:

- RR vectors: station-year monthly profiles from monthly_rainfall
- Ensemble vectors: monthly values from all 5 ensemble members
- Primary score: count of months where RR equals any ensemble member
  (after rounding both values to 2 decimal places)
- Tie-breaker: higher overlap-month count
- Compatibility fields: cosine and adjusted score are still stored
  for diagnostics and historical comparability

The cell below runs this matcher interactively on a bounded slice so the
notebook stays fast. Use the SLURM scripts for full-scale matching.

In [ ]:
# Setup Parquet roots for RR, filtered ensemble, and similarity datasets.

import os
from pathlib import Path

from src.rainfall_rescue_sqlite.ingest import default_db_path
from src.rainfall_rescue_sqlite.ensemble_ingest import default_ensemble_db_path
from src.rainfall_rescue_sqlite.parquet_ingest import default_rainfall_rescue_parquet_root
from src.rainfall_rescue_sqlite.parquet_similarity import default_comparison_parquet_root

# Metadata matching uses only sources that passed transcription QC and were
# deduplicated by the full-scale transcription-QC merge.
rr_dataset_root = default_rainfall_rescue_parquet_root()
ensemble_dataset_root = Path(os.environ["PDIR"]) / "ensemble_transcriptions_parquet_good"
comparison_root = default_comparison_parquet_root()

# Legacy SQLite paths kept for downstream legacy sections in this notebook.
db_path = default_db_path()
ensemble_db_path = default_ensemble_db_path()
comparison_db_path = Path(f"{os.getenv('PDIR')}/monthly_similarity.sqlite")

rr_dataset_root, ensemble_dataset_root, comparison_root

In [ ]:
# Build comparison vectors from RR + ensemble parquet datasets.
#
# MEMORY NOTE: this step loads all RR vectors (~285k station-years) and all
# ensemble consensus vectors (~514k files) into Python memory to compute
# monthly medians and IQRs before writing them to parquet. On the full dataset
# that requires ~8-16 GB RAM, so it is usually run by the local staged pipeline
# (`scripts/local/submit_local.sh similarity`) rather than interactively.
#
# Set rebuild_vectors = True only if you want to (re)build from scratch in this
# notebook. If comparison_root already contains vectors from a prior pipeline
# run, leave it False and go straight to the matching cell below.

from src.rainfall_rescue_sqlite.parquet_similarity import build_comparison_vectors_parquet

rebuild_vectors = False  # set True to rebuild; leave False if vectors already exist

if rebuild_vectors:
    build_result = build_comparison_vectors_parquet(
        rr_dataset_root=rr_dataset_root,
        ensemble_dataset_root=ensemble_dataset_root,
        comparison_root=comparison_root,
    )
    print(build_result)
else:
    rr_vec_path = comparison_root / "rr_monthly_vectors"
    ens_vec_path = comparison_root / "ensemble_consensus_vectors"
    if rr_vec_path.exists() and ens_vec_path.exists():
        print(f"Skipping build - vectors already present in {comparison_root}")
    else:
        print(
            "WARNING: comparison_root has no vectors yet.\n"
            "Either set rebuild_vectors = True (needs ~8-16 GB RAM for full dataset),\n"
            "or run the local full pipeline first:\n"
            "  scripts/local/submit_local.sh similarity"
        )

In [ ]:
# Run bounded similarity matching against existing comparison vectors.
#
# max_ensemble_queries and max_rr_candidates cap how many vectors are loaded,
# so this cell is safe to run on a workstation regardless of dataset size.
# The matching itself keeps only the RR candidate matrix in RAM (~27 MB for
# 285k candidates) and streams ensemble queries - it does not load all vectors.
#
# For a full-scale run use the local similarity pipeline instead
# (`scripts/local/submit_local.sh similarity`), which parallelises queries
# across shards and merges them into one session.

from src.rainfall_rescue_sqlite.parquet_similarity import run_baseline_matching_parquet

match_result = run_baseline_matching_parquet(
    comparison_root=comparison_root,
    top_k=10,
    min_overlap=10,
    uncertainty_weight=0.15,
    max_ensemble_queries=200,    # remove limits for full-scale run
    max_rr_candidates=20000,
)

print(match_result)

In [ ]:
# Inspect top exact matches from the latest parquet similarity session.

import duckdb

conn = duckdb.connect()
try:
    latest_session = conn.execute(
        f"SELECT MAX(session_id) FROM read_parquet('{comparison_root / 'similarity_sessions' / '*.parquet'}')"
    ).fetchone()[0]

    rows = conn.execute(
        f"""
        SELECT
            m.query_rank,
            m.exact_agreement_count,
            m.adjusted_score,
            m.cosine_similarity,
            m.overlap_months,
            m.ensemble_uncertainty,
            e.file_name,
            e.descriptor,
            r.station_file_id,
            r.year,
            r.location_name
        FROM read_parquet('{comparison_root / 'similarity_matches' / '*.parquet'}') m
        JOIN read_parquet('{comparison_root / 'ensemble_consensus_vectors' / '*.parquet'}') e
          ON e.ensemble_vector_id = m.ensemble_vector_id
        JOIN read_parquet('{comparison_root / 'rr_monthly_vectors' / '*.parquet'}') r
          ON r.rr_vector_id = m.rr_vector_id
        WHERE m.session_id = ?
          AND m.query_rank = 1
          AND m.exact_agreement_count = 11
        ORDER BY m.adjusted_score DESC
        LIMIT 20
        """
        , [latest_session]
    ).fetchall()
finally:
    conn.close()

print(f"Latest session: {latest_session}")
for row in rows:
    print(
        f"rank={row[0]:>2}  exact={row[1]:>2}  "
        f"overlap={row[4]:>2}  score={row[2]:.4f}  "
        f"cos={row[3]:.4f}  "
        f"unc={row[5] if row[5] is not None else 'n/a'}  "
        f"ensemble={row[6]}  rr={row[8]}:{row[9]}  "
        f"loc={row[10] or 'n/a'}"
    )

## Sample example specifiers by fitting category

To eyeball how each kind of fitting result looks, the cell below draws a random
set of ensemble specifiers from the latest combined `ensemble_metadata` session
for a chosen category:

- **`exact_data`** &mdash; exact match against the DATA Rainfall-Rescue records
  (`match_type = 'exact'`; has coordinates).
- **`exact_allsheets`** &mdash; exact match against an ALLSHEETS source sheet
  (`match_type = 'exact_allsheets'`; coordinates present only where the name was
  found in `LeftOverSites.csv`).
- **`approximate`** &mdash; top-3 centroid match (`match_type = 'approximate'`).
- **`none`** &mdash; no match (all metadata NULL).

Set `sample_category` and `n_samples`, then paste any returned specifier into the
single-transcription diagnostic cell above to inspect that case. Set
`random_seed` to a float in `[-1, 1]` for a repeatable draw.


In [ ]:
# Sample example specifiers for a chosen fitting category.
#
# Pick a category and a sample size; the cell draws that many ensemble records
# at random from the latest combined ensemble_metadata session and prints their
# specifiers (file-name stems). Paste a specifier into the single-transcription
# diagnostic cell above to inspect that case.

import duckdb
from pathlib import Path

# --- choose here -----------------------------------------------------------
sample_category = "none"  # exact_data | exact_allsheets | approximate | none
n_samples = 25
random_seed = None  # float in [-1, 1] for a reproducible draw, or None for fresh
# ---------------------------------------------------------------------------

_CATEGORY_FILTER = {
    "exact_data": "match_type = 'exact'",
    "exact_allsheets": "match_type = 'exact_allsheets'",
    "approximate": "match_type = 'approximate'",
    "none": "match_type IS NULL",
}
if sample_category not in _CATEGORY_FILTER:
    raise ValueError(
        f"Unknown category {sample_category!r}; choose from {list(_CATEGORY_FILTER)}"
    )

# Latest combined ensemble_metadata session under the DATA comparison root.
_meta_dir = comparison_root / "ensemble_metadata"
_latest_meta = max(
    _meta_dir.glob("session_*.parquet"), key=lambda p: int(p.stem.split("_")[1])
)
_where = _CATEGORY_FILTER[sample_category]

_con = duckdb.connect()
try:
    if random_seed is not None:
        _con.execute("SELECT setseed(?)", [float(random_seed)])
    _total = _con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{_latest_meta}') WHERE {_where}"
    ).fetchone()[0]
    _rows = _con.execute(
        f"""
        SELECT file_name, matched_location_name, matched_year,
               matched_latitude, matched_longitude
        FROM read_parquet('{_latest_meta}')
        WHERE {_where}
        ORDER BY random()
        LIMIT {int(n_samples)}
        """
    ).fetchall()
finally:
    _con.close()

sample_specifiers = [Path(r[0]).stem for r in _rows]

print(f"Session file : {_latest_meta.name}")
print(f"Category     : {sample_category}  ({_total:,} records total)")
print(f"Random sample of {len(_rows)}:\n")
for _r, _spec in zip(_rows, sample_specifiers):
    _loc = _r[1] or "n/a"
    _year = _r[2] if _r[2] is not None else "n/a"
    _coord = (
        f"lat={_r[3]:.2f} lon={_r[4]:.2f}"
        if _r[3] is not None and _r[4] is not None
        else "no coords"
    )
    print(f"  {_spec}")
    print(f"      loc={_loc}  year={_year}  {_coord}")
